# Conversational RAG with Gradio: Code Flow & Use Cases

## 1. What is RAG?
**Retrieval-Augmented Generation (RAG)** combines:
- **Retriever** → fetches relevant documents from a knowledge base.
- **Generator (LLM)** → produces answers grounded in those documents.

This ensures responses are **accurate, contextual, and traceable**.

---

## 2. Code Flow Overview

### Offline Setup (Data Preparation)
1. **Web Loader** → Fetches webpage content (Cricbuzz RCB results).
2. **Text Splitter** → Breaks content into chunks.
3. **Embeddings** → Converts chunks into vectors (HuggingFace model).
4. **Vector Store (FAISS)** → Stores embeddings for fast retrieval.

### Online Query Flow (User Interaction)
1. **User Prompt** → Typed into Gradio Chat.
2. **Retriever Node** → Finds relevant chunks from FAISS.
3. **Responder Node** → Sends context + question to Groq LLM.
4. **Groq LLM** → Generates grounded answer.
5. **Gradio ChatInterface** → Streams answer back like ChatGPT.

---

## 3. Diagram : Code Flow

In [2]:
![AI Agent Query Processing Flow](./images/AI_Agent_Query_Processing.png)

'[AI' is not recognized as an internal or external command,
operable program or batch file.


# RAG {Graph}

In [ ]:
# -----------------------------
# 0. Setup & Imports
# -----------------------------
import os
from typing import List
from pydantic import BaseModel

from dotenv import load_dotenv
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_groq import ChatGroq   # Groq LLM integration
from langchain_huggingface import HuggingFaceEmbeddings   # HuggingFace embeddings
from langchain_core.documents import Document
from langgraph.graph import StateGraph, END

c:\Users\admin\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


USER_AGENT environment variable not set, consider setting it to identify your requests.


In [4]:
# Load environment variables
load_dotenv()
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

# Initialize Groq LLM
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    api_key=os.environ["GROQ_API_KEY"]
)

In [5]:
# -----------------------------
# 1. Document Loading
# -----------------------------
# Use Cricbuzz RCB results page (static HTML, works with WebBaseLoader)
urls = ["https://www.cricbuzz.com/cricket-team/royal-challengers-bangalore/59/results"]

loader = WebBaseLoader(urls)
docs = loader.load()

print(f"Loaded {len(docs)} documents from Cricbuzz")

Loaded 1 documents from Cricbuzz


In [6]:
# -----------------------------
# 2. Text Splitting & Vector Store
# -----------------------------
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)
split_docs = splitter.split_documents(docs)

embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(split_docs, embedding)
retriever = vectorstore.as_retriever()

print("Retriever Test:", retriever.invoke("RCB recent match score"))

Retriever Test: [Document(id='bcf36d7f-f1c0-4bbb-b712-864e928b7b60', metadata={'source': 'https://www.cricbuzz.com/cricket-team/royal-challengers-bangalore/59/results', 'title': 'Royal Challengers Bengaluru Cricket Team matches, scorecards, news and statistics | Cricbuzz.com', 'description': 'Royal Challengers Bengaluru Cricket Team Latest News & Info, Photo Gallery, Stats, Squad, Ranking, Venues & Cricket Score of all the matches on Cricbuzz.com', 'language': 'en'}, page_content="Royal Challengers Bengaluru Cricket Team matches, scorecards, news and statistics | Cricbuzz.comMenuLive ScoresScheduleArchivesNewsSeriesTeamsVideosRankingsMoreMATCHESNZ vs IRE - Trail by 374ENGW vs INDW - PreviewGT vs RR - PreviewRR vs SRH - RR wonRCB vs GT - RCB wonALLAllLive NowTodayINTERNATIONALICC Men's T20 World Cup Sub Regional Africa Qualifier Group A 2026Botswana vs Ivory Coast 13th MatchSierra Leone vs Rwanda 14th MatchCameroon vs Kenya 15th MatchCameroon vs Mali 16th MatchIvory Coast"), Document(id

In [7]:
# -----------------------------
# 3. Define RAG State
# -----------------------------
class RAGState(BaseModel):
    question: str
    retrieved_docs: List[Document] = []
    answer: str = ""

In [8]:
# -----------------------------
# 4. LangGraph Nodes
# -----------------------------
def retrieve_docs(state: RAGState) -> RAGState:
    docs = retriever.invoke(state.question)
    return RAGState(question=state.question, retrieved_docs=docs)

def generate_answer(state: RAGState) -> RAGState:
    context = "\n\n".join([doc.page_content for doc in state.retrieved_docs])
    prompt = f"Answer the question based on the context.\n\nContext:\n{context}\n\nQuestion: {state.question}"
    response = llm.invoke(prompt)
    return RAGState(
        question=state.question,
        retrieved_docs=state.retrieved_docs,
        answer=response.content
    )

In [9]:
# -----------------------------
# 5. Build LangGraph
# -----------------------------
builder = StateGraph(RAGState)
builder.add_node("retriever", retrieve_docs)
builder.add_node("responder", generate_answer)

builder.set_entry_point("retriever")
builder.add_edge("retriever", "responder")
builder.add_edge("responder", END)

graph = builder.compile()

In [10]:
# -----------------------------
# 6. Run the Agentic RAG
# -----------------------------
if __name__ == "__main__":
    user_question = "RCB status"
    initial_state = RAGState(question=user_question)
    final_state = graph.invoke(initial_state)

    print("\n Final Answer:\n", final_state['answer'])


 Final Answer:
 I couldn't find any information about RCB's status. However, I found that RCB (Royal Challengers Bengaluru) won their match against GT (Gujarat Titans) as per the news on the page: "RCB vs GT - RCB won".


In [11]:
def rag_pipeline(user_question: str):
    initial_state = RAGState(question=user_question)
    final_state = graph.invoke(initial_state)
    return final_state['answer']

demo = gr.Interface(
    fn=rag_pipeline,
    inputs=gr.Textbox(lines=2, placeholder="Ask about RCB recent matches..."),
    outputs="text",
    title="RCB Match Results RAG",
    description="Ask questions like 'recent match score of RCB' or 'last 5 matches of RCB'."
)

if __name__ == "__main__":
    demo.launch()

NameError: name 'gr' is not defined

In [ ]:
import gradio as gr

# Agentic RAG with memory

In [ ]:
# -----------------------------
# 0. Setup & Imports
# -----------------------------
import os
from typing import List
from pydantic import BaseModel

import gradio as gr
from dotenv import load_dotenv
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langgraph.graph import StateGraph, END

# Load environment variables
load_dotenv()
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

# Initialize Groq LLM with streaming enabled
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    api_key=os.environ["GROQ_API_KEY"],
    streaming=True   # enables token-by-token streaming
)

# -----------------------------
# 1. Document Loading
# -----------------------------
urls = ["https://www.cricbuzz.com/cricket-team/royal-challengers-bangalore/59/results"]
loader = WebBaseLoader(urls)
docs = loader.load()

# -----------------------------
# 2. Text Splitting & Vector Store
# -----------------------------
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)
split_docs = splitter.split_documents(docs)

embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(split_docs, embedding)
retriever = vectorstore.as_retriever()

# -----------------------------
# 3. Define RAG State
# -----------------------------
class RAGState(BaseModel):
    question: str
    retrieved_docs: List[Document] = []
    answer: str = ""

# -----------------------------
# 4. LangGraph Nodes
# -----------------------------
def retrieve_docs(state: RAGState) -> RAGState:
    docs = retriever.invoke(state.question)
    return RAGState(question=state.question, retrieved_docs=docs)

def generate_answer(state: RAGState) -> RAGState:
    context = "\n\n".join([doc.page_content for doc in state.retrieved_docs])
    prompt = f"Answer the question based on the context.\n\nContext:\n{context}\n\nQuestion: {state.question}"
    response = llm.invoke(prompt)
    return RAGState(
        question=state.question,
        retrieved_docs=state.retrieved_docs,
        answer=response.content
    )

# -----------------------------
# 5. Build LangGraph
# -----------------------------
builder = StateGraph(RAGState)
builder.add_node("retriever", retrieve_docs)
builder.add_node("responder", generate_answer)

builder.set_entry_point("retriever")
builder.add_edge("retriever", "responder")
builder.add_edge("responder", END)

graph = builder.compile()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6903.38it/s]


In [ ]:
# -----------------------------
# 6. Gradio Chat Interface
# -----------------------------
def rag_chat(user_message, history):
    """Chat-style RAG pipeline with context following"""
    initial_state = RAGState(question=user_message)
    final_state = graph.invoke(initial_state)
    return final_state['answer']

demo = gr.ChatInterface(
    fn=rag_chat,
    title=" RCB Match Results Chat RAG",
    description="Ask about RCB matches, scores, and results. Conversation flows like ChatGPT with context."
)

if __name__ == "__main__":
    demo.launch()

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.
